<a href="https://www.kaggle.com/code/ps6nlb/267600821-analyze-sentiment?scriptVersionId=329444027" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 69.0 MB/s eta 0:00:00


# Import Library

In [2]:
import numpy as np 
import pandas as pd 
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import recall_score, precision_score, accuracy_score, classification_report

# สำรวจข้อมูล Wongnai-Food-Review-Rating-Prediction-Challenge

In [4]:
df = pd.read_csv('/kaggle/input/competitions/wongnai-food-review-rating-prediction-challenge/train_data.csv')
df

,review_body,star_rating
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,2
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,3
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,2
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,4
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,4
...,...,...
39995,รู้จักร้านนี้จากวงใน ร้านอยู่ในอาคารพาณิชย์ตรง...,3
39996,ร้านซูชิอาหารญี่ปุ่น อยู่ตรงสะพานลอย เกษตรประต...,3
39997,"""Cantina Wine Bar & Italian Kitchen"" ร้านเล็กๆ...",4
39998,ร้านเค้กน่ารักๆ ตรงชั้นล่างของห้างเซ็นทรัลลาดพ...,2


In [5]:
df.groupby('star_rating').count()

,review_body
star_rating,
0,415
1,1845
2,12171
3,18770
4,6799


# การจัดเตรียมข้อมูล (Data Preparation)

แปลงตัวแปรเป้าหมาย (Label) จาก star_rating เป็นค่าประเมินความคิดเห็นตามโจทย์ <br> Start_Rating 0-1 แปลงเป็นความคิดเห็นเชิงลบ (neg : 0) <br> Start_Rating 2 แปลงเป็นความคิดเห็นทั่วไป (gen : 1) <br>Start_Rating 3-4 แปลงเป็นความคิดเห็นเชิงบวก (pos : 2)

In [6]:
df_neg = df[(df['star_rating'] == 0) | (df['star_rating'] == 1) ]
df_neg

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,1
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,1
101,ไม่รู้จะใจร้ายไปรึเปล่า แต่ไม่อร่อยอ่ะค่ะ\n\n1...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,1
111,วนเวียนอยู่แถวซอยอารีย์ หาร้านส้มตำทานกันเถอะๆ...,1
...,...,...
39869,เรียกว่าเป็นร้านที่โดยส่วนตัวแล้วค่อนข้างเซอร์...,1
39896,#Bangkok N°329/T982/P11605+(6)\n\nอาทิตย์​ก่อน...,1
39922,การปรับปรุงเมนูและวัตถุดิบจากครั้งล่าสุด ทำให้...,1
39938,สำหรับเรา เจียงเป็นร้านก๋วยเตี๋ยวร้านดังที่ไม่...,1


In [7]:
df_neg.loc[:, ['star_rating']] = 0
df_neg

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0
101,ไม่รู้จะใจร้ายไปรึเปล่า แต่ไม่อร่อยอ่ะค่ะ\n\n1...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0
111,วนเวียนอยู่แถวซอยอารีย์ หาร้านส้มตำทานกันเถอะๆ...,0
...,...,...
39869,เรียกว่าเป็นร้านที่โดยส่วนตัวแล้วค่อนข้างเซอร์...,0
39896,#Bangkok N°329/T982/P11605+(6)\n\nอาทิตย์​ก่อน...,0
39922,การปรับปรุงเมนูและวัตถุดิบจากครั้งล่าสุด ทำให้...,0
39938,สำหรับเรา เจียงเป็นร้านก๋วยเตี๋ยวร้านดังที่ไม่...,0


In [8]:
df_gen = df[df['star_rating'] == 2]
df_gen

,review_body,star_rating
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,2
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,2
7,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,2
22,ระหว่างทางไปเขาค้อ ผมได้คำแนะนำจากเด็กปั๊มให้ม...,2
31,ร้านอาหารเรื่อนปั้นหยา ที่จอดรถกว้างขวางสะดวกส...,2
...,...,...
39989,ร้านข้าวมันไก่ในซอยวัดหนองขาม มาแวะซื้อใส่กล่อ...,2
39990,ร้านอยู่กลางเมกะบางนา ตรงประชาสัมพันธ์ ใต้บันไ...,2
39993,นั่งกิน Bar B Q กันอยู่ มองไปทาง food court เห...,2
39998,ร้านเค้กน่ารักๆ ตรงชั้นล่างของห้างเซ็นทรัลลาดพ...,2


In [9]:
df_gen.loc[:, ['star_rating']] = 1
df_gen

,review_body,star_rating
0,ร้านอาหารใหญ่มากกกกกกก \nเลี้ยวเข้ามาเจอห้องน้...,1
2,ปอเปี๊ยะสด ทุกวันนี้รู้สึกว่าหากินยาก (ร้านที่...,1
7,สารภาพว่าไม่เคยคิดจะไปต่อคิวซื้อมากินเองครับ บ...,1
22,ระหว่างทางไปเขาค้อ ผมได้คำแนะนำจากเด็กปั๊มให้ม...,1
31,ร้านอาหารเรื่อนปั้นหยา ที่จอดรถกว้างขวางสะดวกส...,1
...,...,...
39989,ร้านข้าวมันไก่ในซอยวัดหนองขาม มาแวะซื้อใส่กล่อ...,1
39990,ร้านอยู่กลางเมกะบางนา ตรงประชาสัมพันธ์ ใต้บันไ...,1
39993,นั่งกิน Bar B Q กันอยู่ มองไปทาง food court เห...,1
39998,ร้านเค้กน่ารักๆ ตรงชั้นล่างของห้างเซ็นทรัลลาดพ...,1


In [10]:
df_pos = df[(df['star_rating'] == 3) | (df['star_rating'] == 4) ]
df_pos.loc[:, ['star_rating']] = 2
df_pos

,review_body,star_rating
1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา...,2
3,รัานคัพเค้กในเมืองไทยมีไม่มาก หลายๆคนอาจจะสงสั...,2
4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักร...,2
5,ร้านข้าวต้มกระดูกหมู ปากซอยพัฒนาการ 57 เป็นอีก...,2
6,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอา...,2
...,...,...
39992,ร้านที่ชื่อแปลก ตกแต่งสไตล์loft ได้อย่างลงตัว ...,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2
39995,รู้จักร้านนี้จากวงใน ร้านอยู่ในอาคารพาณิชย์ตรง...,2
39996,ร้านซูชิอาหารญี่ปุ่น อยู่ตรงสะพานลอย เกษตรประต...,2


In [11]:
df1 = pd.concat([df_neg, df_gen, df_pos])
df1

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0
72,ร้าน After You อยู่ในทองหล่อซอย 13 เข้าไปประมา...,0
101,ไม่รู้จะใจร้ายไปรึเปล่า แต่ไม่อร่อยอ่ะค่ะ\n\n1...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0
111,วนเวียนอยู่แถวซอยอารีย์ หาร้านส้มตำทานกันเถอะๆ...,0
...,...,...
39992,ร้านที่ชื่อแปลก ตกแต่งสไตล์loft ได้อย่างลงตัว ...,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2
39995,รู้จักร้านนี้จากวงใน ร้านอยู่ในอาคารพาณิชย์ตรง...,2
39996,ร้านซูชิอาหารญี่ปุ่น อยู่ตรงสะพานลอย เกษตรประต...,2


แบ่งข้อมูล Train Set/Test Set ในอัตราส่วน 80:20 โดยวิธีการสุ่ม (Sample)

In [12]:
df_train = df1.sample(32000)
df_train

,review_body,star_rating
20907,ช๊อกโกแลตแสนอร่อยจาก Nama ประเทศญี่ปุ่น โดย RO...,2
12760,ร้านนี้เป็นอีกหนึ่งทางเลือกของผู้ที่ไปเที่ยวเเ...,2
22710,ร้านนี้อยุ่ใกล้กับที่จอดรถวันละ100 บาท\nของคนท...,1
9289,landing กาแฟสด ไม่ได้มีแต่กาแฟสดแต่เพียงอย่างเ...,1
38903,ร้าน The Duke's มี 3 สาขาในเชียงใหม่มั้งครับ ท...,2
...,...,...
21748,ร้านอุด้งและเทมปุระ มีรายการของทอดให้เลือกหลาย...,1
26192,ได้มีโอกาสมาทานร้านนี้ เนื่องจากมาสัมมนาแล้วเค...,0
39303,ร้านร้อนมาก ควันขโมง เจอเตาถ่านอีกยิ่งร้อน กิน...,0
14414,วันก่อนนิวคุงไปทำธุระในย่านพญาไท ก็รู้สึกอยากก...,0


In [13]:
df_test = df1.drop(df_train.index)
df_test

,review_body,star_rating
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0
137,ได้มีโอกาสกลับไปกินโออิชิราเมนอีกครั้งหลังจากท...,0
202,\n\n\nShabushi สาขารัตนาธิเบศร์ ร้านบุฟเฟต์ชาบ...,0
216,สวัสดีค่ะ เพื่อนๆชาวนักกินทุกคน^^\nไม่ได้เจอกั...,0
...,...,...
39984,ฝากท้องฝากไส้กันไป ออกมานอกพื้นที่ ทานร้านริมท...,2
39988,ร้านเจ้าประจำเวลานึกถึงกาแฟอีกร้าน แต่ถ้าเป็นช...,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2


แยกคำที่มีในพจนานุกรมภาษาไทยลงใน List โดยใช้ไลบรารี่ pythainlp

In [14]:
def token_thai(text):
    tokens = word_tokenize(text, engine='newmm')
    return [word for word in tokens if word.strip() != '']

In [15]:
token_thai('ทดสอบแยกคำในภาษาไทย')

['ทดสอบ', 'แยก', 'คำ', 'ใน', 'ภาษาไทย']

นำ Data Set ไปวิเคราะห์ตาราง TF-IDF เพื่อหาคำเพื่อปราฏบ่อยที่สุดในเอกสาร

In [16]:
vectorizer = TfidfVectorizer(tokenizer=token_thai, token_pattern=None, max_features=10000, min_df= 5, max_df=0.9)
x_train = vectorizer.fit_transform(df_train['review_body'])

In [17]:
y_train = df_train['star_rating']
y_train

20907    2
12760    2
22710    1
9289     1
38903    2
        ..
21748    1
26192    0
39303    0
14414    0
15176    1
Name: star_rating, Length: 32000, dtype: int64

In [18]:
x_test = vectorizer.transform(df_test['review_body'])
x_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 679940 stored elements and shape (8000, 10000)>

In [19]:
y_test = df_test['star_rating']
y_test

39       0
102      0
137      0
202      0
216      0
        ..
39984    2
39988    2
39991    2
39994    2
39996    2
Name: star_rating, Length: 8000, dtype: int64

# การประมวลผลการพยากรณ์โดยใช้ Machine Learning / การประเมินโมเดล Classification

In [20]:
model_nai = MultinomialNB()
model_nai.fit(x_train, y_train)
y_pred = model_nai.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.15      0.01      0.01       489
           1       0.61      0.06      0.11      2417
           2       0.65      0.99      0.79      5094

    accuracy                           0.65      8000
   macro avg       0.47      0.35      0.30      8000
weighted avg       0.61      0.65      0.53      8000



In [21]:
model_svc = LinearSVC()
model_svc.fit(x_train, y_train)
y_pred = model_svc.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.63      0.29      0.40       489
           1       0.54      0.45      0.49      2417
           2       0.76      0.86      0.81      5094

    accuracy                           0.70      8000
   macro avg       0.64      0.53      0.57      8000
weighted avg       0.69      0.70      0.69      8000



In [22]:
model_dt =  DecisionTreeClassifier()
model_dt.fit(x_train, y_train)
y_pred = model_dt.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.17      0.13      0.15       489
           1       0.39      0.40      0.39      2417
           2       0.70      0.70      0.70      5094

    accuracy                           0.58      8000
   macro avg       0.42      0.41      0.41      8000
weighted avg       0.57      0.58      0.57      8000



เลือกใช้ Model ที่มีค่าผลการประเมินที่ดีที่สุดนั้นคือโมเดล LogisticRegression Classification

In [23]:
model_lr = LogisticRegression()
model_lr.fit(x_train, y_train)
y_pred = model_lr.predict(x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.78      0.25      0.38       489
           1       0.58      0.45      0.51      2417
           2       0.76      0.89      0.82      5094

    accuracy                           0.72      8000
   macro avg       0.70      0.53      0.57      8000
weighted avg       0.71      0.72      0.70      8000



In [24]:
y_pred

array([2, 1, 1, ..., 2, 2, 2], shape=(8000,))

หา Feature Impotant จาก Model ที่ดีที่สุด เพื่อหาคำที่มีผลต่อการทำนายของโมเดลมากที่สุด โดยแบ่งเป็น 3 คลาส ได้แก่ ความคิดเห็นเชิงลบ (neg), ความคิดเห็นทั่วไป (gen), ความคิดเห็นเชิงบวก (pos)

In [25]:
df_feature_neg = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[0])})
df_feature_neg.sort_values(by = 'Coef', ascending=False)[0:9]

,Feature,Coef
9850,ไม่,5.629745
9852,ไม่ค่อย,3.845604
2940,จืด,3.082650
9258,แย่,2.848723
9261,แย่มาก,2.846321
5139,พอ,2.831560
4761,ปรับปรุง,2.824054
4179,ธรรมดา,2.689103
3387,ดีกว่า,2.456142


In [26]:
df_feature_gen = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[1])})
df_feature_gen.sort_values(by = 'Coef', ascending=False)[0:9]

,Feature,Coef
1886,กลางๆ,3.226229
9676,ใช้ได้,3.021939
5162,พอใช้ได้,2.352542
4349,นิด,1.821336
9798,ได้มาตรฐาน,1.711732
8125,เท่าที่ควร,1.638730
3384,ดี,1.624613
9996,⭐️⭐️⭐️,1.515176
9500,โปร,1.514311


In [27]:
df_feature_pos = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[2])})
df_feature_pos.sort_values(by = 'Coef', ascending=False)[0:9]

,Feature,Coef
7335,อร่อย,5.803534
3392,ดีมาก,3.309632
8018,เด็ด,3.269252
3031,ชอบ,3.123688
3390,ดีงาม,2.940530
5126,พลาด,2.568820
6125,ลงตัว,2.559620
2029,กำลังดี,2.512152
5312,ฟิน,2.506887


# การทดสอบชุดข้อมูล Test Set

นำข้อมูลจากการทำนายโมเดลมาเปรียบเทียบกับข้อมูลจริงในชุดข้อมูล TestSet

In [28]:
df_ex = df_test.copy()
df_ex['predict'] = y_pred
df_ex

,review_body,star_rating,predict
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0,2
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0,1
137,ได้มีโอกาสกลับไปกินโออิชิราเมนอีกครั้งหลังจากท...,0,1
202,\n\n\nShabushi สาขารัตนาธิเบศร์ ร้านบุฟเฟต์ชาบ...,0,1
216,สวัสดีค่ะ เพื่อนๆชาวนักกินทุกคน^^\nไม่ได้เจอกั...,0,1
...,...,...,...
39984,ฝากท้องฝากไส้กันไป ออกมานอกพื้นที่ ทานร้านริมท...,2,2
39988,ร้านเจ้าประจำเวลานึกถึงกาแฟอีกร้าน แต่ถ้าเป็นช...,2,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2,2


In [29]:
df_ex.to_csv('result1.csv', encoding='utf-8-sig')

กรองข้อมูลเฉพาะโมเดลที่ทำนายถูก

In [30]:
df_ex_true = df_ex[df_ex['star_rating'] == df_ex['predict']]
df_ex_true

,review_body,star_rating,predict
442,กราบขออภัยสำหรับ...คำพูด แต่ผมมีความรู้สึกอย่า...,0,0
687,เคยทานเมื่อปีที่แล้วละเลิกทานไปเลยค่ะ เป็นร้าน...,0,0
928,อีกร้าน franchise โปรดที่ทำเสียความรู้สึก คาดว...,0,0
1054,มาเดิน terminal แล้วรู้สึกไม่รู้จะกินอะไรเบื่อ...,0,0
1089,ร้านนี้อยูในซอยตรงข้ามไปรษณีย์กลาง เข้ามากินทด...,0,0
...,...,...,...
39984,ฝากท้องฝากไส้กันไป ออกมานอกพื้นที่ ทานร้านริมท...,2,2
39988,ร้านเจ้าประจำเวลานึกถึงกาแฟอีกร้าน แต่ถ้าเป็นช...,2,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2,2


In [31]:
df_ex_true[df_ex_true['predict'] == 2]

,review_body,star_rating,predict
12,เคยไปทานที่ Villa มา คนเยอะมากกกกก รอคิวนานประ...,2,2
13,เหมือนกับรีวิวก่อนๆค่ะ เมนูที่ห้ามพลาดก้อคือ S...,2,2
30,ร้านเกาหลีใกล้รถไฟฟ้าพญาไท อยู่บริเวณปทุมวันรี...,2,2
75,ร้านไอศครีมน่ารักๆ เหมาะกับพาครอบครัวมานั่งทาน...,2,2
82,ถึงแม้เมนูอาหารจะไม่ได้เยอะมาก แต่อาหารทุกจานอ...,2,2
...,...,...,...
39984,ฝากท้องฝากไส้กันไป ออกมานอกพื้นที่ ทานร้านริมท...,2,2
39988,ร้านเจ้าประจำเวลานึกถึงกาแฟอีกร้าน แต่ถ้าเป็นช...,2,2
39991,เป็นร้านนั่งชิวขึ้นชื่อของโซนนี้ที่ต้องมาลองให...,2,2
39994,"""อร่อย ขนมจีบ ซาลาเปา"" ขนมจีบลูกใหญ่ กับซาลาเป...",2,2


กรองข้อมูลเฉพาะโมเดลที่ทำนายผิด

In [32]:
df_ex_fault = df_ex.drop(df_ex_true.index)
df_ex_fault

,review_body,star_rating,predict
39,ร้านข้าวหมูกรอบที่ผมมักฝากท้องไว้\nที่โดดเด่นก...,0,2
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0,1
137,ได้มีโอกาสกลับไปกินโออิชิราเมนอีกครั้งหลังจากท...,0,1
202,\n\n\nShabushi สาขารัตนาธิเบศร์ ร้านบุฟเฟต์ชาบ...,0,1
216,สวัสดีค่ะ เพื่อนๆชาวนักกินทุกคน^^\nไม่ได้เจอกั...,0,1
...,...,...,...
39771,ใช้โปรบัตร 7-11 โอเคมากค่ะ ตัดเงินจากบัตร 69 บ...,2,1
39832,ร้านดี บรรยากาศโอเพ่นแอร์ชิวๆ ตัวร้านไม่ได้ใหญ...,2,1
39937,เป็นร้านที่หลบเข้ามาในซอยย่อยเรียบคลองประปา บร...,2,1
39960,มาซื้อของไปซ่อมบ้าน เลยมีโอกาสได้ฝากท้องกับร้า...,2,1


In [33]:
df_ex_fault[df_ex_fault['predict'] == 1]

,review_body,star_rating,predict
102,ร้านนี้รสชาติใช้ได้ค่ะ อร่อย โดยเฉพาะพวกผัดกระ...,0,1
137,ได้มีโอกาสกลับไปกินโออิชิราเมนอีกครั้งหลังจากท...,0,1
202,\n\n\nShabushi สาขารัตนาธิเบศร์ ร้านบุฟเฟต์ชาบ...,0,1
216,สวัสดีค่ะ เพื่อนๆชาวนักกินทุกคน^^\nไม่ได้เจอกั...,0,1
371,ผมเป็นคนชอบทานก๋วยเตี๋ยวมากครับ...มื้อเช้าและเ...,0,1
...,...,...,...
39771,ใช้โปรบัตร 7-11 โอเคมากค่ะ ตัดเงินจากบัตร 69 บ...,2,1
39832,ร้านดี บรรยากาศโอเพ่นแอร์ชิวๆ ตัวร้านไม่ได้ใหญ...,2,1
39937,เป็นร้านที่หลบเข้ามาในซอยย่อยเรียบคลองประปา บร...,2,1
39960,มาซื้อของไปซ่อมบ้าน เลยมีโอกาสได้ฝากท้องกับร้า...,2,1


# การประยุกต์นำเอาไปใช้งาน โดยการสร้างฟังก์ชัน analyze_sentiment (sentence) และทดสอบข้อความอื่นๆ ที่ไม่มีใน Dataset

In [34]:
def analyze_sentiment (sentence) :
    x_test = vectorizer.transform(pd.Series(sentence))
    y_pred = model_lr.predict(x_test)
    
    if y_pred.item() == 0:
        y_comment = 'ลบ'
    elif y_pred.item() == 1:
        y_comment = 'ทั่วไป'
    else:
        y_comment = 'บวก' 
        
    print(f'ผลการประเมินโดยใช้แบบจำลอง : {y_comment}')

ทดสอบประโยคนอกเหนือจากใน Dataset 5 ตัวอย่าง

In [35]:
# Ex.1 กรณีที่ 1 : ตัวอย่างการแสดงความคิดเห็นชื่นชม | ความคาดหวัง : เชิงบวก
sentence = 'เนื้อย่างพรีเมียมมาก นุ่มละลายในปาก พนักงานดูแลใส่ใจดี เปลี่ยนตะแกรงให้ตลอด คุ้มค่าสมราคาครับ แนะนำเลยไม่ผิดหวังแน่นอน'
analyze_sentiment (sentence)

ผลการประเมินโดยใช้แบบจำลอง : บวก


In [36]:
# EX.3 กรณีที่ 2 : ตัวอย่างการแสดงความคิดเห็นแบบกลางๆ | ความคาดหวัง : เชิงทั่วไป
sentence = 'ร้านตกแต่งสวยดี ถ่ายรูปมุมไหนก็สวย แต่อาหารรสชาติกลางๆ ไม่ได้โดดเด่นอะไรเมื่อเทียบกับร้านอื่น ราคาแอบแรงไปนิดนึง'
analyze_sentiment (sentence)

ผลการประเมินโดยใช้แบบจำลอง : ทั่วไป


In [37]:
# EX.3 กรณีที่ 3 : ตัวอย่างการแสดงความคิดเห็นแสดงอาการไม่พอใจ | ความคาดหวัง : เชิงลบ
sentence = 'อาหารเค็มจัด รอนานเป็นชั่วโมงแถมได้ออเดอร์ผิด พนักงานหน้าหงิกมากตอนทวงถาม เสียดายเงินและเวลาสุดๆ ไม่มาเหยียบอีกแล้ว'
analyze_sentiment (sentence)

ผลการประเมินโดยใช้แบบจำลอง : ลบ


In [38]:
# EX.4 กรณีที่ 4 ตัวอย่างการแสดงความคิดเห็นแบบประชดประชันในบริบทคำพูดในภาษาไทย | ความคาดหวัง : เชิงลบ
sentence = 'อร่อยจนลืมกลืนเลยครับ เพราะเนื้อเหนียวเป็นยางรถยนต์ เคี้ยวจนกรามค้าง ดีจริงๆ ดีที่ฟันไม่หัก'
analyze_sentiment (sentence)

ผลการประเมินโดยใช้แบบจำลอง : บวก


In [39]:
# EX.5 กรณีที่ 5 ตัวอย่างการใช้ภาษาวัยรุ่น | ความคาดหวัง : เชิงลบ
sentence = 'รสชาติคือจะเครซี่ นอยด์อ่า บ้งมาก ช็อตฟีลสุดๆ ไม่จึ้งเลยแม่ง'
analyze_sentiment (sentence)

ผลการประเมินโดยใช้แบบจำลอง : บวก
